# Clase 4: Árboles de Decisión, Ensembles y Gradient Boosting

## Ejemplos prácticos para la clase

---
## 1. Árbol de Decisión: entrenamiento y visualización

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report

# Cargar datos
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Entrenar un árbol de decisión
arbol = DecisionTreeClassifier(
    max_depth=4,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
arbol.fit(X_train, y_train)

# Evaluar
y_pred = arbol.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=['Maligno', 'Benigno'])}")

In [ ]:
# Visualizar el árbol (gran ventaja de este modelo)

plt.figure(figsize=(20, 10))
plot_tree(arbol,
          feature_names=data.feature_names,
          class_names=["Maligno", "Benigno"],
          filled=True,
          rounded=True,
          fontsize=9)
plt.title("Árbol de Decisión: Diagnóstico de Cáncer de Mama", fontsize=14)
plt.tight_layout()
plt.show()

print("Cada nodo muestra:")
print("  - La pregunta (feature <= valor)")
print("  - Gini: impureza (0 = puro, 0.5 = máxima mezcla)")
print("  - Samples: cuántas muestras hay")
print("  - Value: [cantidad maligno, cantidad benigno]")
print("  - Class: la predicción del nodo")

---
## 2. Índice Gini e Information Gain

In [ ]:
# Demostración: cómo se calcula Gini y Entropía

def gini(proporciones):
    return 1 - sum(p**2 for p in proporciones)

def entropia(proporciones):
    return -sum(p * np.log2(p) for p in proporciones if p > 0)

# Distintos escenarios
escenarios = [
    ("50/50 (máxima mezcla)", [0.5, 0.5]),
    ("70/30", [0.7, 0.3]),
    ("90/10 (bastante puro)", [0.9, 0.1]),
    ("100/0 (totalmente puro)", [1.0, 0.0]),
]

print(f"{'Escenario':<30} {'Gini':>8} {'Entropía':>10}")
print("-" * 50)
for nombre, props in escenarios:
    g = gini(props)
    e = entropia(props)
    print(f"{nombre:<30} {g:>8.4f} {e:>10.4f}")

print(f"\nCuanto menor el valor, más 'puro' el nodo (mejor separación).")

In [ ]:
# Visualizar Gini y Entropía en función de la proporción

p_vals = np.linspace(0.01, 0.99, 100)
gini_vals = [gini([p, 1-p]) for p in p_vals]
entropia_vals = [entropia([p, 1-p]) for p in p_vals]

plt.figure(figsize=(8, 5))
plt.plot(p_vals, gini_vals, color="steelblue", linewidth=2, label="Gini")
plt.plot(p_vals, entropia_vals, color="#e74c3c", linewidth=2, label="Entropía")
plt.xlabel("Proporción de la clase positiva")
plt.ylabel("Impureza")
plt.title("Gini vs Entropía: ambas miden cuánto están mezcladas las clases")
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Overfitting en Árboles: efecto de la profundidad

In [ ]:
# Comparar árboles con distinta profundidad

profundidades = [1, 2, 3, 5, 7, 10, 15, None]
resultados_train = []
resultados_test = []

print(f"{'Profundidad':>12} | {'Acc Train':>10} | {'Acc Test':>10} | {'Hojas':>6}")
print("-" * 50)

for prof in profundidades:
    arbol_temp = DecisionTreeClassifier(max_depth=prof, random_state=42)
    arbol_temp.fit(X_train, y_train)
    
    acc_train = accuracy_score(y_train, arbol_temp.predict(X_train))
    acc_test = accuracy_score(y_test, arbol_temp.predict(X_test))
    n_hojas = arbol_temp.get_n_leaves()
    
    resultados_train.append(acc_train)
    resultados_test.append(acc_test)
    
    prof_str = str(prof) if prof else "Sin límite"
    print(f"{prof_str:>12} | {acc_train:>10.4f} | {acc_test:>10.4f} | {n_hojas:>6}")

print(f"\nObservá cómo sin límite el train llega a 1.0 (memorizó) pero test no mejora.")

In [ ]:
# Gráfico del overfitting

plt.figure(figsize=(10, 5))
x_labels = [str(p) if p else "∞" for p in profundidades]
plt.plot(range(len(profundidades)), resultados_train, "o-", color="steelblue",
         linewidth=2, label="Train Accuracy")
plt.plot(range(len(profundidades)), resultados_test, "o-", color="#e74c3c",
         linewidth=2, label="Test Accuracy")

plt.xticks(range(len(profundidades)), x_labels)
plt.xlabel("Profundidad máxima")
plt.ylabel("Accuracy")
plt.title("Overfitting: a mayor profundidad, más memorización")
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

# Zona de overfitting
plt.axvspan(5, len(profundidades)-1, alpha=0.1, color="red", label="Zona de overfitting")

plt.tight_layout()
plt.show()

---
## 4. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Entrenar Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("RANDOM FOREST (100 árboles):")
print(classification_report(y_test, y_pred_rf, target_names=["Maligno", "Benigno"]))

In [ ]:
# Importancia de features: ¿qué variables son más útiles para predecir?

importancias = pd.Series(rf.feature_importances_, index=X.columns)
top_features = importancias.sort_values(ascending=True).tail(10)

plt.figure(figsize=(10, 6))
top_features.plot(kind="barh", color="steelblue")
plt.xlabel("Importancia")
plt.title("Top 10 Features más Importantes (Random Forest)")
plt.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

print("Las features más importantes para predecir si un tumor es maligno o benigno:")
for feat, imp in top_features.sort_values(ascending=False).items():
    print(f"  {feat}: {imp:.4f}")

---
## 5. XGBoost

In [ ]:
# pip install xgboost (si no está instalado)

try:
    from xgboost import XGBClassifier
    
    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss"
    )
    xgb.fit(X_train, y_train)
    y_pred_xgb = xgb.predict(X_test)
    
    print("XGBOOST:")
    print(classification_report(y_test, y_pred_xgb, target_names=["Maligno", "Benigno"]))
except ImportError:
    print("XGBoost no está instalado. Instalalo con: pip install xgboost")

---
## 6. LightGBM

In [ ]:
# pip install lightgbm (si no está instalado)

try:
    from lightgbm import LGBMClassifier
    
    lgbm = LGBMClassifier(
        n_estimators=200,
        max_depth=-1,
        num_leaves=31,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1
    )
    lgbm.fit(X_train, y_train)
    y_pred_lgbm = lgbm.predict(X_test)
    
    print("LIGHTGBM:")
    print(classification_report(y_test, y_pred_lgbm, target_names=["Maligno", "Benigno"]))
except ImportError:
    print("LightGBM no está instalado. Instalalo con: pip install lightgbm")

---
## 7. Comparación de todos los modelos

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Escalar para Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Definir todos los modelos
modelos = {
    "Logistic Regression": (LogisticRegression(max_iter=1000), True),
    "Decision Tree (depth=5)": (DecisionTreeClassifier(max_depth=5, random_state=42), False),
    "Random Forest": (RandomForestClassifier(n_estimators=100, random_state=42), False),
}

# Agregar XGBoost y LightGBM si están disponibles
try:
    from xgboost import XGBClassifier
    modelos["XGBoost"] = (XGBClassifier(n_estimators=100, random_state=42, eval_metric="logloss"), False)
except ImportError:
    pass

try:
    from lightgbm import LGBMClassifier
    modelos["LightGBM"] = (LGBMClassifier(n_estimators=100, random_state=42, verbose=-1), False)
except ImportError:
    pass

# Evaluar cada modelo
resultados = []
print("=" * 65)
print(f"{'Modelo':<25} {'CV F1 (mean)':>12} {'CV F1 (std)':>12} {'Test F1':>10}")
print("=" * 65)

for nombre, (modelo, necesita_escalar) in modelos.items():
    if necesita_escalar:
        modelo.fit(X_train_scaled, y_train)
        cv_scores = cross_val_score(modelo, X_train_scaled, y_train, cv=5, scoring="f1")
        y_pred_m = modelo.predict(X_test_scaled)
    else:
        modelo.fit(X_train, y_train)
        cv_scores = cross_val_score(modelo, X_train, y_train, cv=5, scoring="f1")
        y_pred_m = modelo.predict(X_test)
    
    from sklearn.metrics import f1_score
    test_f1 = f1_score(y_test, y_pred_m)
    resultados.append({"modelo": nombre, "cv_f1": cv_scores.mean(), "test_f1": test_f1})
    
    print(f"{nombre:<25} {cv_scores.mean():>12.4f} {cv_scores.std():>12.4f} {test_f1:>10.4f}")

print("=" * 65)

In [ ]:
# Visualizar la comparación

res_df = pd.DataFrame(resultados)

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(res_df))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], res_df["cv_f1"], width,
               label="CV F1 (promedio)", color="steelblue", alpha=0.8)
bars2 = ax.bar([i + width/2 for i in x], res_df["test_f1"], width,
               label="Test F1", color="#e74c3c", alpha=0.8)

ax.set_ylabel("F1-Score")
ax.set_title("Comparación de Modelos")
ax.set_xticks(x)
ax.set_xticklabels(res_df["modelo"], rotation=15, ha="right")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(0.9, 1.0)

plt.tight_layout()
plt.show()

---
## 8. Hyperparameter Tuning con GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

# Buscar los mejores hiperparámetros para Random Forest
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 10, None],
    "min_samples_leaf": [1, 3, 5],
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=0
)

grid.fit(X_train, y_train)

print(f"Mejores hiperparámetros: {grid.best_params_}")
print(f"Mejor F1 (CV): {grid.best_score_:.4f}")

# Evaluar en test con el mejor modelo
mejor_modelo = grid.best_estimator_
y_pred_mejor = mejor_modelo.predict(X_test)
print(f"\nF1 en test: {f1_score(y_test, y_pred_mejor):.4f}")
print(f"\nReporte:")
print(classification_report(y_test, y_pred_mejor, target_names=["Maligno", "Benigno"]))

In [ ]:
# Tuning de XGBoost con GridSearchCV

try:
    from xgboost import XGBClassifier
    
    param_grid_xgb = {
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.1, 0.3],
    }
    
    grid_xgb = GridSearchCV(
        XGBClassifier(random_state=42, eval_metric="logloss"),
        param_grid_xgb,
        cv=5,
        scoring="f1",
        n_jobs=-1,
        verbose=0
    )
    
    grid_xgb.fit(X_train, y_train)
    
    print(f"XGBoost - Mejores hiperparámetros: {grid_xgb.best_params_}")
    print(f"XGBoost - Mejor F1 (CV): {grid_xgb.best_score_:.4f}")
    
    y_pred_xgb_best = grid_xgb.best_estimator_.predict(X_test)
    print(f"XGBoost - F1 en test: {f1_score(y_test, y_pred_xgb_best):.4f}")
except ImportError:
    print("XGBoost no está instalado.")

---
## 9. Pipeline completo con datos propios (casas.csv)

In [ ]:
# Ejemplo de regresión con ensembles usando nuestro dataset de casas

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

casas = pd.read_csv("casas.csv")
casas_encoded = pd.get_dummies(casas, columns=["barrio"], drop_first=True)

X = casas_encoded.drop("precio", axis=1)
y = casas_encoded["precio"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelos_reg = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=4, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
}

print("=" * 60)
print(f"{'Modelo':<25} {'RMSE (USD)':>12} {'R²':>10}")
print("=" * 60)

for nombre, modelo in modelos_reg.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    print(f"{nombre:<25} {rmse:>12,.0f} {r2:>10.4f}")

print("=" * 60)
print(f"\nNota: el dataset es chico (15 casas), los ensembles brillan con más datos.")

---
## Resumen

| Modelo | Tipo | Ventaja | Desventaja |
|--------|------|---------|------------|
| Árbol de Decisión | Individual | Interpretable, visual | Overfitting |
| Random Forest | Bagging | Robusto, buen default | Menos interpretable |
| XGBoost | Boosting | Máximo rendimiento | Requiere tuning |
| LightGBM | Boosting | Rápido, datasets grandes | Requiere tuning |

### Receta práctica:
1. Baseline simple (Logistic Regression / Linear Regression)
2. Random Forest (buen rendimiento con poco esfuerzo)
3. XGBoost / LightGBM + GridSearchCV (máximo rendimiento)